# INF2179 Classification Challenge

# 1. Import Libraries

In [27]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

# 2. Load data


In [28]:
train = pd.read_csv("../data/train_data.csv")
test = pd.read_csv("../data/test_data.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("\nColumns:")
print(train.columns.tolist())


Train shape: (18788, 10)
Test shape: (4698, 10)

Columns:
['Clothing ID', 'Age', 'Title', 'Review Text', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name']


In [29]:
train.head()

,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,1092,31,Versatile,I passed over this dress online thinking i'd n...,5,1,0,General,Dresses,Dresses
1,406,33,"Such a unique, flattering swimsuit!",I am so excited about this swimsuit. i have ne...,5,1,0,Initmates,Intimate,Swim
2,1008,31,"The perfect skirt for all year, every year.","To start: i'm 135 lbs, 5'7"", 34c, 28 jeans, si...",5,1,0,General Petite,Bottoms,Skirts
3,836,59,Love this blouse,I really like this blouse a lot. very very eas...,5,1,0,General,Tops,Blouses
4,1030,39,So crazy comfortable,I haven't worn jeans in a year because i felt ...,5,1,0,General,Bottoms,Jeans


# 3. Basic data check

In [30]:
# Check for missing values
print("\nMissing values in train:")
print(train[["Title", "Review Text", "Rating", "Recommended IND"]].isnull().sum())

print("\nMissing values in test:")
print(test[["Title", "Review Text"]].isnull().sum())

print("\nRecommended IND distribution:")
print(train["Recommended IND"].value_counts())

print("\nRating distribution:")
print(train["Rating"].value_counts().sort_index())



Missing values in train:
Title              3062
Review Text         700
Rating                0
Recommended IND       0
dtype: int64

Missing values in test:
Title          748
Review Text    145
dtype: int64

Recommended IND distribution:
Recommended IND
1    15467
0     3321
Name: count, dtype: int64

Rating distribution:
Rating
1      689
2     1248
3     2251
4     4060
5    10540
Name: count, dtype: int64


# 4. Text cleaning function

In [31]:
def clean_text(text):
    text = str(text).lower()                    # lowercase
    text = re.sub(r"[^a-z\s]", " ", text)      # remove punctuation/numbers
    text = re.sub(r"\s+", " ", text).strip()   # remove extra spaces
    return text

# 5. Apply text cleaning

In [32]:
train["Title"] = train["Title"].fillna("").apply(clean_text)
train["Review Text"] = train["Review Text"].fillna("").apply(clean_text)

test["Title"] = test["Title"].fillna("").apply(clean_text)
test["Review Text"] = test["Review Text"].fillna("").apply(clean_text)

# Combine title and review text
train["CombinedText"] = train["Title"] + " " + train["Review Text"]
test["CombinedText"] = test["Title"] + " " + test["Review Text"]

print("\nSample cleaned text:")
print(train["CombinedText"].head(3))



Sample cleaned text:
0    versatile i passed over this dress online thin...
1    such a unique flattering swimsuit i am so exci...
2    the perfect skirt for all year every year to s...
Name: CombinedText, dtype: object


# 6. Task 1: Recommendation Prediction (model development)

In [33]:
# Use cleaned combined text (Title + Review Text) to predict recommendation
X_rec = train["CombinedText"]
y_rec = train["Recommended IND"]

In [34]:
# Split training data for model development
X_train_rec, X_valid_rec, y_train_rec, y_valid_rec = train_test_split(
    X_rec,
    y_rec,
    test_size=0.2,
    random_state=42,
    stratify=y_rec
)

In [35]:
# TF-IDF is used to convert the cleaned review text into numerical features.
# We include both unigrams and bigrams to capture single words and short phrases.
# sublinear_tf=True reduces the effect of very frequent words,
# max_features=12000 controls feature size,
# and min_df=2 removes extremely rare terms to reduce noise.

In [36]:
# TF-IDF for Task 1
tfidf_rec = TfidfVectorizer(sublinear_tf=True,
    max_features=12000,
    ngram_range=(1, 2),
    min_df=2,
)

X_train_rec_tfidf = tfidf_rec.fit_transform(X_train_rec)
X_valid_rec_tfidf = tfidf_rec.transform(X_valid_rec)

In [37]:
# Logistic Regression is chosen because it performs well on sparse text features.
# C=2.0 allows slightly weaker regularization,
# max_iter=2000 ensures the model converges,
# random_state=42 keeps results reproducible,
# class_weight="balanced" helps handle class imbalance,
# and liblinear is suitable for binary classification.

In [38]:
# Logistic Regression for Task 1
model_rec = LogisticRegression(C=2.0,
    max_iter=2000,
    random_state=42,
    class_weight="balanced",
    solver="liblinear"
)

model_rec.fit(X_train_rec_tfidf, y_train_rec)

LogisticRegression(C=2.0, class_weight='balanced', max_iter=2000,
                   random_state=42, solver='liblinear')

In [39]:
# Validation prediction (used only for model development)
pred_valid_rec = model_rec.predict(X_valid_rec_tfidf)

print("\nTask 1 - Validation Classification Report")
print(classification_report(y_valid_rec, pred_valid_rec))

valid_task1_accuracy = accuracy_score(y_valid_rec, pred_valid_rec)
valid_task1_weighted_f1 = f1_score(y_valid_rec, pred_valid_rec, average="weighted")

print("Task 1 - Validation Accuracy:", valid_task1_accuracy)
print("Task 1 - Validation Weighted F1:", valid_task1_weighted_f1)


Task 1 - Validation Classification Report
              precision    recall  f1-score   support

           0       0.70      0.84      0.76       664
           1       0.96      0.92      0.94      3094

    accuracy                           0.91      3758
   macro avg       0.83      0.88      0.85      3758
weighted avg       0.92      0.91      0.91      3758

Task 1 - Validation Accuracy: 0.9079297498669505
Task 1 - Validation Weighted F1: 0.9111717397024822


# 7. Final Task 1 model: train on training data, evaluate on testing data

In [40]:
# Retrain the best Task 1 model on the full training dataset,
# then evaluate it on the provided testing dataset using the true labels.

In [41]:
tfidf_rec_final = TfidfVectorizer(
    sublinear_tf=True,
    max_features=12000,
    ngram_range=(1, 2),
    min_df=2,
)

X_rec_all_tfidf = tfidf_rec_final.fit_transform(train["CombinedText"])
X_rec_test_tfidf = tfidf_rec_final.transform(test["CombinedText"])

model_rec_final = LogisticRegression(
    C=2.0,
    max_iter=2000,
    random_state=42,
    class_weight="balanced",
    solver="liblinear"
)

model_rec_final.fit(X_rec_all_tfidf, y_rec)

test_pred_rec = model_rec_final.predict(X_rec_test_tfidf)

print("\nTask 1 - Testing Classification Report")
print(classification_report(test["Recommended IND"], test_pred_rec))

task1_accuracy = accuracy_score(test["Recommended IND"], test_pred_rec)
task1_weighted_f1 = f1_score(test["Recommended IND"], test_pred_rec, average="weighted")

print("Task 1 - Testing Accuracy:", task1_accuracy)
print("Task 1 - Testing Weighted F1:", task1_weighted_f1)

submission_task1 = pd.DataFrame({
    "Recommended IND": test_pred_rec
})

submission_task1.to_csv("task1_recommendation_predictions.csv", index=False)

print("\nTask 1 submission file saved: task1_recommendation_predictions.csv")
print(submission_task1["Recommended IND"].value_counts())


Task 1 - Testing Classification Report
              precision    recall  f1-score   support

           0       0.69      0.86      0.76       851
           1       0.97      0.91      0.94      3847

    accuracy                           0.90      4698
   macro avg       0.83      0.88      0.85      4698
weighted avg       0.92      0.90      0.91      4698

Task 1 - Testing Accuracy: 0.9037888463175819
Task 1 - Testing Weighted F1: 0.9076592177005742

Task 1 submission file saved: task1_recommendation_predictions.csv
Recommended IND
1    3641
0    1057
Name: count, dtype: int64


In [42]:
print("\nTask 1 predictions preview:")
print(submission_task1.head())


Task 1 predictions preview:
   Recommended IND
0                1
1                1
2                1
3                0
4                1


# 8. Task 2: Star Rating Prediction (model development)

For Task 2, the cleaned combined text is used to predict the exact star rating from 1 to 5.

In [43]:
X_rate = train["CombinedText"]
y_rate = train["Rating"]

In [44]:
# Split training data for model development
X_train_rate, X_valid_rate, y_train_rate, y_valid_rate = train_test_split(
    X_rate,
    y_rate,
    test_size=0.2,
    random_state=42,
    stratify=y_rate
)

In [45]:
# TF-IDF is again used to transform the cleaned combined text into numerical features.
# Unigrams and bigrams help capture both individual words and short expressions.
# Compared with Task 1, min_df=3 is slightly stricter to remove more rare terms,
# which helps reduce sparsity for the more difficult 5-class prediction task.

In [46]:
# TF-IDF for Task 2
tfidf_rate = TfidfVectorizer(
    sublinear_tf=True,
    max_features=10000,
    ngram_range=(1, 2),
    min_df=3,
)

X_train_rate_tfidf = tfidf_rate.fit_transform(X_train_rate)
X_valid_rate_tfidf = tfidf_rate.transform(X_valid_rate)

In [47]:
# Logistic Regression is used as a strong baseline for multi-class text classification.
# C=1.0 provides standard regularization,
# max_iter=2000 helps ensure convergence,
# random_state=42 improves reproducibility,
# and lbfgs is an appropriate solver for multi-class classification.

In [48]:
# Logistic Regression for Task 2
model_rate = LogisticRegression(
    C=1.0,
    max_iter=2000,
    random_state=42,
    solver="lbfgs",

)

model_rate.fit(X_train_rate_tfidf, y_train_rate)


LogisticRegression(max_iter=2000, random_state=42)

In [49]:
# Validation prediction (used only for model development)
pred_valid_rate = model_rate.predict(X_valid_rate_tfidf)

print("\nTask 2 - Validation Classification Report")
print(classification_report(y_valid_rate, pred_valid_rate))

valid_task2_accuracy = accuracy_score(y_valid_rate, pred_valid_rate)

print("Task 2 - Validation Accuracy:", valid_task2_accuracy)


Task 2 - Validation Classification Report
              precision    recall  f1-score   support

           1       0.54      0.11      0.18       138
           2       0.39      0.20      0.26       250
           3       0.43      0.41      0.42       450
           4       0.48      0.35      0.41       812
           5       0.76      0.93      0.84      2108

    accuracy                           0.67      3758
   macro avg       0.52      0.40      0.42      3758
weighted avg       0.63      0.67      0.63      3758

Task 2 - Validation Accuracy: 0.6657796700372539


# 9. Final Task 2 model: train on training data, evaluate on testing data

In [50]:
# Retrain the best Task 2 model on the full training dataset,
# then evaluate it on the provided testing dataset using the true labels.

In [51]:
tfidf_rate_final = TfidfVectorizer(
    sublinear_tf=True,
    max_features=10000,
    ngram_range=(1, 2),
    min_df=3,
)

X_rate_all_tfidf = tfidf_rate_final.fit_transform(train["CombinedText"])
X_rate_test_tfidf = tfidf_rate_final.transform(test["CombinedText"])

model_rate_final = LogisticRegression(
    C=1.0,
    max_iter=2000,
    random_state=42,
    solver="lbfgs",
)

model_rate_final.fit(X_rate_all_tfidf, y_rate)

test_pred_rate = model_rate_final.predict(X_rate_test_tfidf)

print("\nTask 2 - Testing Classification Report")
print(classification_report(test["Rating"], test_pred_rate))

task2_accuracy = accuracy_score(test["Rating"], test_pred_rate)

print("Task 2 - Testing Accuracy:", task2_accuracy)

submission_task2 = pd.DataFrame({
    "Rating": test_pred_rate
})

submission_task2.to_csv("task2_rating_predictions.csv", index=False)

print("\nTask 2 submission file saved: task2_rating_predictions.csv")
print(submission_task2["Rating"].value_counts().sort_index())


Task 2 - Testing Classification Report
              precision    recall  f1-score   support

           1       0.44      0.14      0.22       153
           2       0.47      0.23      0.31       317
           3       0.48      0.41      0.44       620
           4       0.46      0.35      0.40      1017
           5       0.75      0.93      0.83      2591

    accuracy                           0.66      4698
   macro avg       0.52      0.41      0.44      4698
weighted avg       0.62      0.66      0.63      4698

Task 2 - Testing Accuracy: 0.6604938271604939

Task 2 submission file saved: task2_rating_predictions.csv
Rating
1      50
2     154
3     527
4     778
5    3189
Name: count, dtype: int64


In [52]:
print("\nTask 2 predictions preview:")
print(submission_task2.head())


Task 2 predictions preview:
   Rating
0       5
1       5
2       4
3       3
4       5
